In [ ]:
import torchvision
from torchvision import transforms

train_dataset_raw = torchvision.datasets.CIFAR10(
    root='./data', train=True, download=True
)
test_dataset_raw = torchvision.datasets.CIFAR10(
    root='./data', train=False, download=True
)


In [ ]:
import numpy as np
import pandas as pd

In [ ]:
class_names = train_dataset_raw.classes

train_images = train_dataset_raw.data
test_images = test_dataset_raw.data
train_labels = np.array(train_dataset_raw.targets)
test_labels = np.array(test_dataset_raw.targets)

# flatten images so that each object is represented as a vector
train_flat = train_images.reshape(len(train_images), -1)
test_flat = test_images.reshape(len(test_images), -1)

train_data = pd.DataFrame(train_flat)
train_data['label'] = train_labels

test_data = pd.DataFrame(test_flat)
test_data['label'] = test_labels

train_data.head()


In [ ]:
import torch.nn as nn

class Net(nn.Module):
    def __init__(self):
        super(Net, self).__init__()

        # define layers and activation function that your model will have
        self.fc1 = nn.Linear(3072, 256)  # Input: 32*32*3 = 3072 flattened pixels
        self.fc2 = nn.Linear(256, 64)
        self.fc3 = nn.Linear(64, 1)       # Output: 1 neuron for binary classification
        self.relu = nn.ReLU()
        self.sigmoid = nn.Sigmoid()       # Sigmoid for probability output


    def forward(self, x):

        # define a flow of input through your layers
        x = self.relu(self.fc1(x))
        x = self.relu(self.fc2(x))
        x = self.sigmoid(self.fc3(x))     # Output probability of class 1

        return x

In [ ]:
import torch

# Set up device for GPU acceleration
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

if torch.cuda.is_available():
    print(f"\n--- GPU Information ---")
    print(f"GPU Name: {torch.cuda.get_device_name(0)}")
    print(f"GPU Count: {torch.cuda.device_count()}")
    print(f"Current GPU Index: {torch.cuda.current_device()}")
    print(f"GPU Memory Allocated: {torch.cuda.memory_allocated(0) / 1024**2:.2f} MB")
    print(f"GPU Memory Cached: {torch.cuda.memory_reserved(0) / 1024**2:.2f} MB")
    print(f"GPU Total Memory: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.2f} GB")
    print(f"CUDA Version: {torch.version.cuda}")
    print(f"cuDNN Version: {torch.backends.cudnn.version()}")
else:
    print("\nNo GPU available, using CPU.")

In [ ]:
import torch
from torch.utils.data import Dataset, DataLoader

In [ ]:
class CIFAR10Dataset(Dataset):
    def __init__(self, data):
        '''
        params:
            data (pd.DataFrame) â€” dataframe with flattened CIFAR-10 images and column `label`
        '''

        # load the data and
        # split it into features and target variable
        # and convert both to np.array
        # please do not change names of variables self.X and self.y,
        # it is important for the checks below to work correctly

        # features
        self.X = data.drop(columns=['label']).values
        # target variable
        self.y = data['label'].values

    def __len__(self):
        '''
        method that returns the number of objects in a dataset.
        This method is used by dataloader to generate batches of data
        '''
        return len(self.X)

    def __getitem__(self, idx):
        '''
        method that, given an index idx, returns the dataset object
        corresponding to the index.
        This method is used by dataloader to form batches of data
        params:
            idx: index of an element of the data
        '''

        # - get an object of data by index idx;
        # - normalize features (divide each pixel by 255.);
        # - convert features and target to tensor
        features = self.X[idx] / 255.
        target = self.y[idx]
        
        return torch.tensor(features, dtype=torch.float32), torch.tensor(target, dtype=torch.long)

In [ ]:
cifar_train_dataset = CIFAR10Dataset(train_data)
cifar_test_dataset = CIFAR10Dataset(test_data)


In [ ]:
train_loader = torch.utils.data.DataLoader(cifar_train_dataset, batch_size=64, shuffle=True)
test_loader = torch.utils.data.DataLoader(cifar_test_dataset, batch_size=64, shuffle=False)


In [ ]:
def evaluate(model, loader, criterion):
    '''
    args:
        model - our neural network model
        loader â€” structure which yields batches of test data
        criterion - loss function from `torch.nn` module
    '''

    # arrays for storing loss values, network predictions and true values
    losses = []
    y_pred_list = []
    y_true_list = []

    model.eval()

    for X_batch, y_batch in loader:
        X_batch, y_batch = X_batch.to(device), y_batch.to(device)

        # this disables gradient computations to save time and memory
        # we don't need gradients on test data
        with torch.no_grad():
            # getting our model's predictions on current batch
            y_pred = model(X_batch)

            # calculate loss function
            loss = criterion(y_pred, y_batch)
            losses.append(loss.item())


        # convert outputs of your network into class number
        y_pred = torch.argmax(y_pred, dim=1).tolist()

        # save for accuracy calculation
        y_pred_list.extend(y_pred)
        y_true_list.extend(y_batch.tolist())

    # calculate accuracy score based on y_pred_list and y_true_list
    accuracy = sum(1 for p, t in zip(y_pred_list, y_true_list) if p == t) / len(y_true_list)

    return np.mean(losses), accuracy

In [ ]:
import matplotlib.pyplot as plt
from torch.utils.data import random_split

# ============================================================
# Hyperparameters (Optuna-tunable later)
# ============================================================
LEARNING_RATE = 1e-2
NUM_EPOCHS = 80
BATCH_SIZE = 128
USE_BATCH_NORM = True        # True / False
DROPOUT_RATE = 0.3
HIDDEN_SIZES = (1024, 512, 256, 128)
WEIGHT_DECAY = 1e-4
EARLY_STOP_PATIENCE = 10

# Optimizer: uncomment the one you want to use
OPTIMIZER_NAME = 'sgd_momentum'
# OPTIMIZER_NAME = 'adam'
# OPTIMIZER_NAME = 'adamw'

# ============================================================
# Train / Eval / Test split (80/20 from original training set)
# ============================================================
train_size = int(0.8 * len(cifar_train_dataset))
eval_size = len(cifar_train_dataset) - train_size
train_subset, eval_subset = random_split(
    cifar_train_dataset, [train_size, eval_size],
    generator=torch.Generator().manual_seed(42),
)

train_loader = DataLoader(train_subset, batch_size=BATCH_SIZE, shuffle=True)
eval_loader = DataLoader(eval_subset, batch_size=BATCH_SIZE, shuffle=False)
# test_loader is already defined in a previous cell

# ============================================================
# Model definition
# ============================================================

class FlexibleNet(nn.Module):
    """
    Fully-connected network with configurable:
      - hidden layer sizes (and thus number of layers)
      - dropout rate
      - batch normalisation on/off
    All arguments map 1-to-1 to Optuna-tunable hyperparameters.
    """
    def __init__(self, hidden_sizes, use_batch_norm, dropout_rate):
        super().__init__()
        layers = []
        in_features = 3072  # CIFAR-10 flattened: 32*32*3
        for h in hidden_sizes:
            layers.append(nn.Linear(in_features, h))
            if use_batch_norm:
                layers.append(nn.BatchNorm1d(h))
            layers.append(nn.ReLU())
            if dropout_rate > 0:
                layers.append(nn.Dropout(dropout_rate))
            in_features = h
        layers.append(nn.Linear(in_features, 10))
        self.net = nn.Sequential(*layers)

    def forward(self, x):
        return self.net(x)


def build_optimizer(model, optimizer_name, lr, weight_decay):
    """Return optimizer. Easy to extend for Optuna."""
    if optimizer_name == 'sgd_momentum':
        return torch.optim.SGD(model.parameters(), lr=lr,
                               momentum=0.9, weight_decay=weight_decay)
    elif optimizer_name == 'adam':
        return torch.optim.Adam(model.parameters(), lr=lr,
                                weight_decay=weight_decay)
    elif optimizer_name == 'adamw':
        return torch.optim.AdamW(model.parameters(), lr=lr,
                                 weight_decay=weight_decay)
    else:
        raise ValueError(f"Unknown optimizer: {optimizer_name}")


# ============================================================
# Training loop with early stopping
# ============================================================

model = FlexibleNet(HIDDEN_SIZES, USE_BATCH_NORM, DROPOUT_RATE).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = build_optimizer(model, OPTIMIZER_NAME, LEARNING_RATE, WEIGHT_DECAY)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=NUM_EPOCHS)

train_losses = []
eval_losses = []
best_eval_loss = float('inf')
best_epoch = 0
epochs_without_improvement = 0

for epoch in range(NUM_EPOCHS):
    # --- Train ---
    model.train()
    running_loss = 0.0
    num_batches = 0
    for batch_X, batch_y in train_loader:
        batch_X, batch_y = batch_X.to(device), batch_y.to(device)
        optimizer.zero_grad()
        outputs = model(batch_X)
        loss = criterion(outputs, batch_y)
        loss.backward()
        optimizer.step()
        running_loss += loss.item()
        num_batches += 1
    scheduler.step()
    avg_train_loss = running_loss / num_batches
    train_losses.append(avg_train_loss)

    # --- Evaluate ---
    eval_loss, eval_acc = evaluate(model, eval_loader, criterion)
    eval_losses.append(eval_loss)

    # --- Early stopping ---
    if eval_loss < best_eval_loss:
        best_eval_loss = eval_loss
        best_epoch = epoch
        epochs_without_improvement = 0
        best_model_state = model.state_dict().copy()
    else:
        epochs_without_improvement += 1

    if (epoch + 1) % 5 == 0 or epoch == 0:
        print(f"Epoch {epoch+1:3d}/{NUM_EPOCHS}  "
              f"Train Loss: {avg_train_loss:.4f}  "
              f"Eval Loss: {eval_loss:.4f}  Eval Acc: {eval_acc:.4f}")

    if epochs_without_improvement >= EARLY_STOP_PATIENCE:
        print(f"\nEarly stopping at epoch {epoch+1} "
              f"(best eval loss {best_eval_loss:.4f} at epoch {best_epoch+1})")
        break

# Restore best model
model.load_state_dict(best_model_state)
print(f"\nRestored best model from epoch {best_epoch+1}")

# ============================================================
# Plot train & eval loss
# ============================================================
epochs_ran = len(train_losses)
plt.figure(figsize=(10, 5))
plt.plot(range(1, epochs_ran + 1), train_losses, label='Train Loss')
plt.plot(range(1, epochs_ran + 1), eval_losses, label='Eval Loss')
plt.axvline(x=best_epoch + 1, color='grey', linestyle='--', alpha=0.7,
            label=f'Best epoch ({best_epoch+1})')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('Train vs Eval Loss')
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()

# ============================================================
# Final evaluation: train, eval, test accuracy
# ============================================================
train_loss_final, train_acc = evaluate(model, train_loader, criterion)
eval_loss_final, eval_acc = evaluate(model, eval_loader, criterion)
test_loss_final, test_acc = evaluate(model, test_loader, criterion)

print(f"\n{'='*50}")
print(f"FINAL RESULTS (best model from epoch {best_epoch+1})")
print(f"{'='*50}")
print(f"  {'Split':<8s}  {'Loss':>8s}  {'Accuracy':>8s}")
print(f"  {'Train':<8s}  {train_loss_final:8.4f}  {train_acc:8.4f}")
print(f"  {'Eval':<8s}  {eval_loss_final:8.4f}  {eval_acc:8.4f}")
print(f"  {'Test':<8s}  {test_loss_final:8.4f}  {test_acc:8.4f}")
